# **Calibración Siguiendo Model Acemoglu y Restrepo 2022**

## Notas:
- Esta calibración sigue de cerca la metodología propuesta por A&R (2022)
- Falta usar bases de datos para información sobre productividad y participación de capital.
- No se contemplan hogares.
- Se calibra usando datos ficticios, revisar BAE (informacion industria) y O*NET (información tareas).

## 1. Configuración inicial

In [6]:
import numpy as np
import pandas as pd
import os
from scipy.optimize import fsolve, minimize

if os.getcwd().split('\\')[len(os.getcwd().split('\\'))-1] != "202510-MacroLP-Proyecto":
    os.chdir("..")
print(os.getcwd())

c:\Users\NicolasLozano\OneDrive - Universidad de los andes\UNIVERSIDAD\9. NOVENO SEMESTRE\MACRO AVANZADA LP\202510-MacroLP-Proyecto


## 2. Set-up del modelo

In [7]:
np.random.seed(123)

I = 10  # numero de industrias
G = 4   # tipo de trabajos/tare
tasks = ['cognitiva','socioemocional','juicio','repetitiva']

# Alphas: se normalizan a 1
alpha = np.ones(G+1)
# Se crean los psi de tal forma que sumen 1.
psi_matrix = np.random.dirichlet(alpha, size=I)
# Shares del ingreso
sYi = np.random.dirichlet(np.ones(I))  
# Productividad de cada industria.  
Ai  = np.random.lognormal(mean=0, sigma=0.1, size=I)

df = pd.DataFrame({
    'industry_id': [f'Ind{i+1}' for i in range(I)],
    'sYi': sYi,
    'Ai': Ai,
    'psi_k': psi_matrix[:, 0],
})
for j, task in enumerate(tasks):
    df[f'psi_{task}'] = psi_matrix[:, j+1]

lambda_    = 1.5                       # Elasticidad de
A_k  = 1.0                       # capital‐augmenting tech (normalized)



## 3. Hallamos equilibrio

In [ ]:
def solve_equilibrium(Ag, Ai):
    """
    Solve for equilibrium wages w_g and prices p_i given Ag (G,) and Ai (I,)
    """
    x0 = np.concatenate([np.ones(G), np.ones(I)])  # [w(4), p(10)]
    def eqs(x):
        w = x[:G]
        p = x[G:]
        eqs = []
        # wage equations for each task g
        for g in range(G):
            term = np.sum(
                df['sYi'].values * (Ai * p)**(lambda_ - 1) * psi_matrix[:, g+1]
            )
            rhs = term**(1/lambda_) * Ag[g]**((lambda_ - 1)/lambda_)
            eqs.append(w[g] - rhs)
        # price equations for each industry i
        for i in range(I):
            sum_L = np.sum(w**(1-lambda_) * Ag**(lambda_-1) * psi_matrix[i, 1:])
            rhs_p = (A_k**(lambda_-1) * psi_matrix[i, 0] + sum_L)**(1/(1-lambda_)) / Ai[i]
            eqs.append(p[i] - rhs_p)
        return eqs
    sol = fsolve(eqs, x0)
    return sol[:G], sol[G:]

# -----------------------------------------------------------------------------
# 3) GENERATE SYNTHETIC OBSERVED WAGES
# -----------------------------------------------------------------------------
A_g_true = np.random.lognormal(mean=0, sigma=0.1, size=G)
w_obs, p_obs = solve_equilibrium(A_g_true, df['Ai'].values)

# -----------------------------------------------------------------------------
# 4) CALIBRATION: Recover Ag and Ai from w_obs
# -----------------------------------------------------------------------------
def objective(params):
    Ag = params[:G]
    Ai = params[G:]
    w_mod, _ = solve_equilibrium(Ag, Ai)
    return np.sum((w_mod - w_obs)**2)

# initial guesses
x0 = np.concatenate([np.ones(G), np.ones(I)])

res = minimize(objective, x0, method='L-BFGS-B')
Ag_cal = res.x[:G]
Ai_cal = res.x[G:]

# Report results
print("True A_g vs Calibrated A_g:")
for t, true, cal in zip(tasks, A_g_true, Ag_cal):
    print(f"  {t:15s}: {true:.4f} → {cal:.4f}")

print("\nTrue A_i vs Calibrated A_i:")
for ind, true, cal in zip(df['industry_id'], df['Ai_true'], Ai_cal):
    print(f"  {ind:6s}: {true:.4f} → {cal:.4f}")



KeyError: 'Ai_true'